In [2]:
import os
from getpass import getpass
from langchain_community.vectorstores.chroma import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import MultiVectorRetriever
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate
#from app.sqlite_docstore import SQLiteDocstore
import pickle
from langchain.storage import InMemoryStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain.chains import LLMChain
from dotenv import load_dotenv
from langchain_core.runnables import ConfigurableField
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables.base import RunnableSerializable
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

import json

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv('LANGCHAIN_TRACING_V2')
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ["LANGCHAIN_PROJECT"] = os.getenv('LANGCHAIN_PROJECT')



# 1. Load embeddings, parent_documents, store and retriever

In [3]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./chroma/multivector_chroma_db_001"
)

with open("parent_documents.pkl", "rb") as f:
    parent_documents = pickle.load(f)


store = InMemoryStore()
store.mset([(d.metadata["id"], d) for d in parent_documents])

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key="parent_id",
    search_kwargs={"k": 3} # reminder: top 3 similitud para los chunks resumidos, no para los fallos completos
)



/var/folders/pg/nt5xcr_n6rd6qh7kjj9bnkpr0000gp/T/ipykernel_79436/2953357986.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/Users/Bauti/Desktop/jurisprudence-rag-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/pg/nt5xcr_n6rd6qh7kjj9bnkpr0000gp/T/ipykernel_79436/2953357986.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 

# 2. Define llm & agent worker prompt template

In [5]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.0,)

prompt_template = PromptTemplate.from_template("""
Sos un asistente jurídico.
Usá exclusivamente el contenido de los documentos proporcionados para responder la consulta.
Si encontrás documentos relacionados, proporcioná la información correspondiente de dichos documentos.
No inventes ni infieras información que no esté presente en los textos.
Si no hay jurisprudencia relevante, indicá claramente que no la encontrás.
Tampoco menciones ningún documento si no encontraste juridisprudencia relevante.

Documentos:
{context}

Pregunta:
{question}

Respuesta:

""")

combine_docs_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)

# 3. Tools definition

In [ ]:
@tool
def retriev_documents(question: str):
    """
    Recupera documentos jurídicos relevantes desde una base de datos Chroma utilizando un retriever avanzado.

    Usá esta herramienta cuando necesites buscar fallos, jurisprudencia o documentos legales que puedan contener información útil y específica para responder una consulta jurídica del usuario.

    Esta función es ideal como primer paso para obtener contexto legal fundamentado antes de generar una respuesta. 
    Devuelve únicamente los documentos que, tras un filtrado adicional por un modelo de lenguaje, se consideran relevantes para la pregunta planteada.

    Parámetro:
        question (str): Pregunta jurídica o consulta del usuario.
    """

    retrieved_documents = retriever.invoke(question)

    relevance_prompt = PromptTemplate.from_template("""
    ¿Aporta este texto información relevante para responder la pregunta siguiente?
    Pregunta:
    {question}

    Texto:
    {content}

    Responde únicamente “Sí” o “No”.""")
    relevance_chain = LLMChain(llm=llm, prompt=relevance_prompt)
    filtered_docs = []

    print(retrieved_documents)

    for doc in retrieved_documents:
        verdict = relevance_chain.run(question=question, content=doc.page_content)#["page_content"])
        if verdict.strip().lower().startswith("sí"):
            filtered_docs.append(doc)


    #return [doc.page_content for doc in filtered_docs]

    response = combine_docs_chain.invoke({
        "question": question,
        "context": filtered_docs
    })

    return response

#print("\n🧠 Respuesta del modelo:")
#print(response)

#print("\n📄 Documentos relevantes encontrados:")
#for i, doc in enumerate(filtered_docs):
#    print(f"\n--- Documento {i+1} ---")
#    print("Tribunal:", doc.metadata['Tribunal'])
#    print("Sala:", doc.metadata['Sala'])
#    print("Expediente:", doc.metadata['Expediente'])
#    print("Caratula:", doc.metadata['Caratula'])
#    print("Fecha de Sentencia:", doc.metadata['FechaSentencia'])
        
        

@tool
def final_answer(answer: str, tools_used: list[str]) -> dict:
    """
    Utiliza esta herramienta para proveer una respuesta final al usuario.
    La respuesta debe estar redactada en lenguaje natural.
    'tools_used' es una lista con los nombres de las herramientas utilizadas.
    """
    return {"answer": answer, "tools_used": tools_used}


In [7]:
tools = [retriev_documents, final_answer]#, filter_retrieved_documents, generate_answer_from_documents, final_answer]
tools

[StructuredTool(name='retriev_documents', description='Recupera documentos jurídicos relevantes desde una base de datos Chroma utilizando un retriever avanzado.\n\nUsá esta herramienta cuando necesites buscar fallos, jurisprudencia o documentos legales que puedan contener información útil y específica para responder una consulta jurídica del usuario.\n\nEsta función es ideal como primer paso para obtener contexto legal fundamentado antes de generar una respuesta. \nDevuelve únicamente los documentos que, tras un filtrado adicional por un modelo de lenguaje, se consideran relevantes para la pregunta planteada.\n\nParámetro:\n    question (str): Pregunta jurídica o consulta del usuario.', args_schema=<class 'langchain_core.utils.pydantic.retriev_documents'>, func=<function retriev_documents at 0x138f313a0>),
 StructuredTool(name='final_answer', description="Utiliza esta herramienta para proveer una respuesta final al usuario.\nLa respuesta debe estar redactada en lenguaje natural.\n'tool

## 3.1 name2tool mapping

In [8]:
name2tool = {tool.name: tool.func for tool in tools}
name2tool

{'retriev_documents': <function __main__.retriev_documents(question: str)>,
 'final_answer': <function __main__.final_answer(answer: str, tools_used: list[str]) -> dict>}

# 4. Agent Exectuor definition

In [9]:

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "Eres un asistente legal experto en análisis de fallos judiciales y documentos jurídicos. "
            "Cuando respondas preguntas de usuarios, puedes utilizar las herramientas disponibles para buscar y filtrar información relevante en la base de datos de fallos. "
            "Primero, intenta usar las herramientas provistas. Luego, utiliza el 'scratchpad' para registrar el resultado de cada herramienta utilizada. "
            "Si ya cuentas con suficiente información en el scratchpad para responder la pregunta, "
            "llama a la herramienta 'final_answer' y proporciona la respuesta al usuario usando esa herramienta. "
            "Evita inventar información o responder sin evidencia de los documentos."
        )
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [10]:


class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")
        )

    def invoke(self, input: str) -> dict:
        count = 0
        agent_scratchpad = []
        end_loop = False
        while count < self.max_iterations and not end_loop:

            # Debug prints
            #print(f'[DEBUG] input >> {input}')
            #print(f'[DEBUG] chat_history >> {self.chat_history}')
            #print(f'[DEBUG] agent_scratchpad >> {agent_scratchpad}')
            tool_call = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })

            #print(f'[DEBUG] tool_call >>> {tool_call}')
            agent_scratchpad.append(tool_call) # tool_call tiene la tool que se llamó, los argumentos y un id de la llamada a la tool
            #print(f'[DEBUG] agent_scratchpad >>> {agent_scratchpad}')
            
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_calls[0]["id"]

            if tool_name == "retriev_documents" and "question" in tool_args:
                    tool_args["question"] = input # piso lo generado por la LLM como posible pregunta para el argumento
                                                  # {TO DO} --> mejorar el prompt para que genere mejor el argumento de la tool call

            print(f'[DEBUG] tool_name >>> {tool_name}')
            print(f'[DEBUG] tool_args >>> {tool_args}')

            tool_out = name2tool[tool_name](**tool_args)
            #print(f'[DEBUG] tool_out >>> {tool_out}')

            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            #print(f'[DEBUG] tool_exec >>> {tool_exec}')
            agent_scratchpad.append(tool_exec)
            #print(f'[DEBUG] agent_scratchpad >>> {agent_scratchpad}')

            print(f"{count}: {tool_name}({tool_args})")
            count += 1

            if tool_name == "final_answer":
                end_loop = True


        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])

        return json.dumps(tool_out)

In [11]:
agent_executor = CustomAgentExecutor()

In [12]:
result = agent_executor.invoke(input="¿Existe algún fallo que trate la inconstitucionalidad de la ley de prenda?")
json.loads(result)['answer']

[DEBUG] tool_name >>> retriev_documents
[DEBUG] tool_args >>> {'question': '¿Existe algún fallo que trate la inconstitucionalidad de la ley de prenda?'}
[Document(metadata={'id': 'ad76614a-9dc6-4fc0-ad62-55b28672cce1', 'producer': 'openhtmltopdf.com; modified using iText 2.1.7 by 1T3XT', 'creator': '', 'creationdate': '2025-04-09T09:00:04-03:00', 'source': '/var/folders/pg/nt5xcr_n6rd6qh7kjj9bnkpr0000gp/T/tmpxy0o97yq.pdf', 'file_path': '/var/folders/pg/nt5xcr_n6rd6qh7kjj9bnkpr0000gp/T/tmpxy0o97yq.pdf', 'total_pages': 6, 'format': 'PDF 1.7', 'title': 'Despacho COM 14267/2024/CA002 - SE RESUELVE RECHAZAR LA APELACION', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-09T09:00:04-03:00', 'trapped': '', 'modDate': "D:20250409090004-03'00'", 'creationDate': "D:20250409090004-03'00'", 'page': 0, 'Tribunal': 'CAMARA COMERCIAL - SALA C', 'Expediente': 'COM 014267/2024/CA002', 'Caratula': 'BASSI, GRACIELA NORA c/ SOCIEDAD ITALIANA DE BENEFICENCIA EN BUENOS AIRES s/AMPARO', 'Fech

/var/folders/pg/nt5xcr_n6rd6qh7kjj9bnkpr0000gp/T/ipykernel_79436/3134614850.py:26: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  relevance_chain = LLMChain(llm=llm, prompt=relevance_prompt)
/var/folders/pg/nt5xcr_n6rd6qh7kjj9bnkpr0000gp/T/ipykernel_79436/3134614850.py:34: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  verdict = relevance_chain.run(question=question, content=doc.page_content)#["page_content"])


0: retriev_documents({'question': '¿Existe algún fallo que trate la inconstitucionalidad de la ley de prenda?'})
[DEBUG] tool_name >>> final_answer
[DEBUG] tool_args >>> {'answer': 'Sí, existe un fallo relevante que trata la inconstitucionalidad de la ley de prenda. En el caso "HSBC Bank Argentina S.A. c/García, Dora Claudia s/secuestro prendario", la Cámara Comercial analiza la inconstitucionalidad de la ley de prenda en relación con los derechos de los consumidores. La corte concluye que el régimen de la prenda registral es incompatible con las disposiciones de la Ley de Defensa del Consumidor (Ley 24.240). Se argumenta que el secuestro directo de un bien sin la debida audiencia del deudor consumidor contradice los principios básicos del derecho de consumo, lo que lleva a considerar que la ley de prenda debe entenderse modificada por la ley de defensa del consumidor, priorizando la protección de los derechos de los consumidores.', 'tools_used': ['functions.retriev_documents']}
1: fin

'Sí, existe un fallo relevante que trata la inconstitucionalidad de la ley de prenda. En el caso "HSBC Bank Argentina S.A. c/García, Dora Claudia s/secuestro prendario", la Cámara Comercial analiza la inconstitucionalidad de la ley de prenda en relación con los derechos de los consumidores. La corte concluye que el régimen de la prenda registral es incompatible con las disposiciones de la Ley de Defensa del Consumidor (Ley 24.240). Se argumenta que el secuestro directo de un bien sin la debida audiencia del deudor consumidor contradice los principios básicos del derecho de consumo, lo que lleva a considerar que la ley de prenda debe entenderse modificada por la ley de defensa del consumidor, priorizando la protección de los derechos de los consumidores.'